# islide — M3 polygon drawing demo

M3 adds **drawing** to the annotation overlay: a hand-drawn polygon lands
in the same `annotations` set as an imported one — identical shape model,
identical renderer, identical `read_crop` story.

How to draw:

1. Click somewhere on the canvas (so the view has keyboard focus) and
   press **A** — the cursor becomes a crosshair and the status bar says
   `Polygon: click to add points — A saves, Esc cancels`.
2. **Left-click** each vertex. A drag still pans (in both modes) and
   wheel zoom stays live, so the draft tracks the view.
3. Press **A** again to save, or **Esc** to cancel the draft.

On save, the view sends the *open* ring (no redundant closing position),
in unclamped level-0 px, as `last_polygon`; Python normalizes it with the
same shared ring helper the GeoJSON importer uses (≥ 3 points, nonzero
area) and appends a `polygon` shape. Degenerate drafts (0–2 clicks,
collinear points) are discarded with a warning — `annotations` is
untouched.

> **Note:** drawing needs the built JupyterLab/notebook extension
> (`frontend/`). Without it the cells below still run (the `annotations`
> trait and `read_crop`), but the A/click gestures have no view to drive.


## 1. Open the slide

The slide opens on a background thread; `wait()` blocks until it's ready.


In [ ]:
from pathlib import Path

from islide import SlideViewer

slide = next(
    (p for base in (Path.cwd(), Path.cwd().parent)
     for p in [base / "data" / "testslide.tiff"]
     if p.exists()),
    None,
)
assert slide, "data/testslide.tiff not found"

v = SlideViewer(str(slide))
v.wait()
v


## 2. Draw a polygon

Click the canvas, press **A**, left-click ~4 vertices around the slide
center `(18691, 36611)`, then press **A** again to save. (Press **Esc**
at any point to cancel a draft.) The view stays put: the saved ring is
sent as `last_polygon`, and the appended shape shows up in
`v.annotations`.


In [ ]:
# After the A / click... / A sequence:
# `v.last_polygon` holds the last saved ring (open, unclamped level-0 px)

v.last_polygon

# and the appended polygon shape (fresh id, null styling):

v.annotations


## 3. Read the crop under the drawn polygon

The polygon is just a shape in `annotations` — e.g. read back the tissue
under it (bounding box, padded, at level 0):


In [ ]:
if not v.annotations:
    raise RuntimeError(
        "no annotations yet — draw one first (A, click the vertices, A)"
    )

ring = v.annotations[-1]["points"][0]
xs = [p[0] for p in ring]
ys = [p[1] for p in ring]
pad = 256
w, h = v.meta["dimensions"]
bbox = (
    max(0, min(xs) - pad),
    max(0, min(ys) - pad),
    min(w, max(xs) + pad),
    min(h, max(ys) + pad),
)
img = v.read_crop(bbox)
img


## 4. Clear

`clear_annotations()` removes everything — imported *and* drawn shapes
(together; per-shape deletion is M4).


In [ ]:
v.clear_annotations()
v.annotations
